# Project 1: Predicting Credit Card Default Risk

This notebook uses the **UCI Default of Credit Card Clients** dataset to examine whether customer demographic information, billing amounts, and repayment history can be used to predict whether a client will default on the next payment.

This notebook does four main things:

1. finds and extracts the raw UCI zip file  
2. loads the Excel dataset and checks its structure  
3. cleans the data and saves a modeling-ready CSV  
4. trains a baseline logistic regression model and reports performance  


## 1. Install and import packages

If you are running this in **Google Colab**, upload the file  
`default+of+credit+card+clients.zip`  
to the notebook first.

This notebook will try to find that zip file automatically.


In [ ]:
# If you are in Colab and xlrd is not installed, run this cell.
!pip -q install xlrd


In [ ]:
from pathlib import Path
import zipfile
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)


## 2. Find the raw zip file and extract it

In [ ]:
def find_file(filename: str, start_dir: Path) -> Path:
    matches = list(start_dir.rglob(filename))
    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename}. Upload it to this notebook or place it under {start_dir}."
        )
    return matches[0]

base_dir = Path.cwd()
zip_filename = "default+of+credit+card+clients.zip"

zip_path = find_file(zip_filename, base_dir)
extract_dir = base_dir / "credit_default_raw"
extract_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

xls_path = find_file("default of credit card clients.xls", extract_dir)

print("Zip file found at:", zip_path)
print("Excel file extracted to:", xls_path)


## 3. Load the raw dataset

In [ ]:
# The true header starts on the second row, so use header=1
df = pd.read_excel(xls_path, header=1)

print("Shape:", df.shape)
df.head()


In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)


In [ ]:
print("Missing values by column:")
print(df.isna().sum())

print("\nDuplicate IDs:")
print(df["ID"].duplicated().sum())

print("\nTarget distribution:")
print(df["default payment next month"].value_counts(dropna=False))
print(df["default payment next month"].value_counts(normalize=True, dropna=False))


## 4. Clean the dataset

For modeling, this notebook:
- renames the target column to `default_next_month`
- converts column names to lowercase
- treats `sex`, `education`, and `marriage` as categorical variables
- keeps `id` as a record identifier, but does not use it as a predictor


In [ ]:
cleaned_df = df.rename(columns={"default payment next month": "default_next_month"}).copy()
cleaned_df.columns = [col.lower() for col in cleaned_df.columns]

# small recoding for unusual category values
cleaned_df["education"] = cleaned_df["education"].replace({0: 4, 5: 4, 6: 4})
cleaned_df["marriage"] = cleaned_df["marriage"].replace({0: 3})

cat_cols = ["sex", "education", "marriage"]
for col in cat_cols:
    cleaned_df[col] = cleaned_df[col].astype("category")

print("Cleaned shape:", cleaned_df.shape)
cleaned_df.head()


In [ ]:
print("Cleaned data types:")
print(cleaned_df.dtypes)

print("\nMissing values after cleaning:")
print(cleaned_df.isna().sum().sort_values(ascending=False).head(10))


In [ ]:
processed_dir = base_dir / "processed_data"
processed_dir.mkdir(exist_ok=True)

cleaned_csv_path = processed_dir / "credit_default_cleaned.csv"
cleaned_df.to_csv(cleaned_csv_path, index=False)

print("Saved cleaned dataset to:", cleaned_csv_path)


## 5. Build a baseline prediction pipeline

This baseline model uses:
- **categorical variables**: `sex`, `education`, `marriage`
- **numerical variables**: credit limit, age, repayment status, bill amounts, and payment amounts

The target variable is `default_next_month`.


In [ ]:
target = "default_next_month"

num_cols = [
    "limit_bal", "age",
    "pay_0", "pay_2", "pay_3", "pay_4", "pay_5", "pay_6",
    "bill_amt1", "bill_amt2", "bill_amt3", "bill_amt4", "bill_amt5", "bill_amt6",
    "pay_amt1", "pay_amt2", "pay_amt3", "pay_amt4", "pay_amt5", "pay_amt6"
]

X = cleaned_df[cat_cols + num_cols].copy()
y = cleaned_df[target].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=2000))
    ]
)


In [ ]:
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall:", round(recall_score(y_test, y_pred), 4))
print("F1:", round(f1_score(y_test, y_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))


In [ ]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


In [ ]:
results = pd.DataFrame({
    "actual": y_test.values,
    "predicted": y_pred,
    "predicted_probability": y_prob
})

results.head(10)


## 6. Short interpretation

This notebook shows that the cleaned dataset can be loaded, processed, and used in a basic machine learning pipeline for predicting credit card default. The final model here is only a **baseline logistic regression**, but it is enough for a proof-of-concept pipeline check for Project 1.
